In [0]:
# %sql
# -- 1. Aseguramos que la tabla no tenga basura previa
# DROP TABLE IF EXISTS workspace.formula_1.bronze_sessions;

In [0]:
# %sql
# SELECT * FROM read_files('/Volumes/workspace/formula_1/formula_1/sessions', multiLine => true) LIMIT 5

In [0]:
%sql
-- 1. Crear la tabla SOLO si no existe
-- Usamos TBLPROPERTIES para soportar nombres de columnas con espacios (como tus 'Opcion 1')
CREATE TABLE IF NOT EXISTS workspace.formula_1.bronze_sessions
USING DELTA
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT get_json_object(raw_json, '$.session_name')) AS total_sessions,
    COUNT(DISTINCT get_json_object(raw_json, '$.session_type')) AS total_sessions_types,
    COUNT(DISTINCT get_json_object(raw_json, '$.location')) AS total_locations,
    COUNT(DISTINCT get_json_object(raw_json, '$.circuit_short_name')) AS total_circuit_short_names,
    COUNT(DISTINCT get_json_object(raw_json, '$.country_code')) AS total_country_codes,
    MIN(get_json_object(raw_json, '$.date_start')) AS min_date,
    MAX(get_json_object(raw_json, '$.date_end')) AS max_date
FROM
    workspace.formula_1.bronze_sessions

In [0]:
%sql
COPY INTO workspace.formula_1.bronze_sessions
FROM (
  SELECT 
    to_json(struct(*)) AS raw_json,
    month,
    year,
    _metadata.file_modification_time AS file_metadata_modification_time,
    _metadata.file_path AS file_metadata_path
  FROM '/Volumes/workspace/formula_1/formula_1/sessions'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'false');

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT get_json_object(raw_json, '$.session_name')) AS total_sessions,
    COUNT(DISTINCT get_json_object(raw_json, '$.session_type')) AS total_sessions_types,
    COUNT(DISTINCT get_json_object(raw_json, '$.location')) AS total_locations,
    COUNT(DISTINCT get_json_object(raw_json, '$.circuit_short_name')) AS total_circuit_short_names,
    COUNT(DISTINCT get_json_object(raw_json, '$.country_code')) AS total_country_codes,
    MIN(get_json_object(raw_json, '$.date_start')) AS min_date,
    MAX(get_json_object(raw_json, '$.date_end')) AS max_date
FROM
    workspace.formula_1.bronze_sessions

In [0]:
%sql
SELECT 
* 
FROM 
workspace.formula_1.bronze_sessions
ORDER BY
file_metadata_modification_time DESC
LIMIT 5;